In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from antares_client.search import get_by_id, get_thumbnails
from astroquery.mast import Observations
from astroquery.esa.euclid import Euclid
from astropy.coordinates import SkyCoord
from astropy.io import fits
import astropy.units as u
from io import BytesIO
from PIL import Image, ImageDraw
import numpy as np
import requests
import os

In [ ]:
def get_decals_jpg(ra, dec, size=100, layer="ls-dr10-grz", pixscale=0.262):
    """Fetch DECaLS DR10 colour JPG cutout."""
    url = (f"https://www.legacysurvey.org/viewer/cutout.jpg"
           f"?ra={ra}&dec={dec}&layer={layer}&pixscale={pixscale}&size={size}")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.content

def get_ps1_color_cutout(ra, dec, size=120, output_size=256):
    """Fetch PS1 color cutout. Prefer grz, fall back to gri.
    size = cutout in PS1 pixels (0.25"/pix), so 120 = 30 arcsec on sky.
    """
    url = (f"https://ps1images.stsci.edu/cgi-bin/ps1filenames.py"
           f"?ra={ra}&dec={dec}&filters=griz&type=stack")
    lines = requests.get(url, timeout=30).text.strip().split('\n')
    files = {}
    for line in lines[1:]:
        parts = line.split()
        if len(parts) >= 8:
            files[parts[4]] = parts[7]
    if all(f in files for f in 'grz'):
        red, green, blue, label = files['z'], files['r'], files['g'], 'grz'
    elif all(f in files for f in 'gri'):
        red, green, blue, label = files['i'], files['r'], files['g'], 'gri'
    else:
        return None, None
    url = (f"https://ps1images.stsci.edu/cgi-bin/fitscut.cgi"
           f"?ra={ra}&dec={dec}&size={size}&format=jpg&output_size={output_size}"
           f"&red={red}&green={green}&blue={blue}")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.content, label

def get_hst_cutout(ra, dec, fov_deg=0.003, size=256):
    """Fetch HST cutout via CDS HiPS2FITS, trying multiple band maps."""
    for hips in ['CDS/P/HST/wideV', 'CDS/P/HST/color', 'CDS/P/HST/I',
                 'CDS/P/HST/R', 'CDS/P/HST/V', 'CDS/P/HST/B',
                 'CDS/P/HST/SDSSr', 'CDS/P/HST/SDSSz']:
        url = (f"https://alasky.cds.unistra.fr/hips-image-services/hips2fits"
               f"?hips={hips}&width={size}&height={size}"
               f"&fov={fov_deg}&projection=TAN&coordsys=icrs"
               f"&ra={ra}&dec={dec}&format=jpg")
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200 and len(r.content) > 5000:
                return r.content
        except Exception:
            continue
    return None

def get_euclid_cutout(ra, dec, radius_arcmin=0.1):
    """Fetch Euclid cutout. Try color (VIS+J+H), fall back to VIS grayscale."""
    coord = SkyCoord(ra, dec, unit='deg')
    rad_deg = radius_arcmin / 60.0

    # Find mosaics at this position
    query = (f"SELECT file_path, filter_name FROM q1.mosaic_product "
             f"WHERE INTERSECTS(CIRCLE('ICRS',{ra},{dec},{rad_deg}), fov)=1")
    results = Euclid.launch_job(query).get_results()
    if len(results) == 0:
        return None, None

    filters = {row['filter_name']: row['file_path'] for row in results}

    def fetch_array(fpath):
        tmp = os.path.join(os.getcwd(), '_euclid_cutout.fits')
        Euclid.get_cutout(file_path=fpath, coordinate=coord,
                         radius=radius_arcmin * u.arcmin, output_file=tmp)
        data = fits.getdata(tmp).astype(float)
        os.unlink(tmp)
        return data

    def norm(arr):
        p1, p99 = np.percentile(arr, [1, 99])
        return np.clip((arr - p1) / max(p99 - p1, 1e-10), 0, 1)

    def to_png(pil_img):
        buf = BytesIO()
        pil_img.save(buf, format='PNG')
        return buf.getvalue()

    # Try color: H=red, J=green, VIS=blue
    if all(k in filters for k in ['VIS', 'J', 'H']):
        try:
            arrays = {k: fetch_array(filters[k]) for k in ['VIS', 'J', 'H']}
            # Resize VIS (0.1"/pix) to match NISP (0.3"/pix) if needed
            target = arrays['H'].shape
            for k in arrays:
                if arrays[k].shape != target:
                    arrays[k] = np.array(Image.fromarray(
                        arrays[k].astype(np.float32), mode='F'
                    ).resize((target[1], target[0]), Image.LANCZOS))
            rgb = np.stack([norm(arrays['H']), norm(arrays['J']),
                           norm(arrays['VIS'])], axis=-1)
            return to_png(Image.fromarray((rgb * 255).astype(np.uint8))), 'VIS+J+H'
        except Exception:
            pass

    # Fall back to single band (prefer VIS)
    for band in ['VIS', 'Y', 'J', 'H']:
        if band in filters:
            try:
                data = norm(fetch_array(filters[band]))
                return to_png(Image.fromarray((data * 255).astype(np.uint8))), band
            except Exception:
                continue

    return None, None


def get_jwst_cutout(ra, dec, fov_deg=0.003, size=256):
    """Fetch JWST cutout via CDS HiPS2FITS, trying multiple band maps."""
    for hips in ['ESAVO/P/JWST/NIRCam_Imaging', 'CDS/P/JWST/F444W',
                 'CDS/P/JWST/F200W', 'CDS/P/JWST/F150W',
                 'CDS/P/JWST/F115W', 'CDS/P/JWST/EPO']:
        url = (f"https://alasky.cds.unistra.fr/hips-image-services/hips2fits"
               f"?hips={hips}&width={size}&height={size}"
               f"&fov={fov_deg}&projection=TAN&coordsys=icrs"
               f"&ra={ra}&dec={dec}&format=jpg")
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200 and len(r.content) > 5000:
                return r.content
        except Exception:
            continue
    return None

def ps1_gr_color(ra, dec, radius_arcsec=2.0):
    """Query PS1 DR2 for nearest source g-r color via MAST TAP."""
    radius_deg = radius_arcsec / 3600.0
    query = (f"SELECT TOP 1 gmeanpsfmag, rmeanpsfmag FROM dbo.meanobjectview "
             f"WHERE 1=CONTAINS(POINT('ICRS', ramean, decmean), "
             f"CIRCLE('ICRS', {ra}, {dec}, {radius_deg}))")
    url = 'https://mast.stsci.edu/vo-tap/api/v0.1/ps1dr2/sync'
    r = requests.get(url, params={'REQUEST': 'doQuery', 'LANG': 'ADQL',
                                  'FORMAT': 'csv', 'QUERY': query}, timeout=30)
    r.raise_for_status()
    lines = r.text.strip().split('\n')
    if len(lines) < 2:
        return None, None, None
    vals = lines[1].split(',')
    g = float(vals[0]) if vals[0] and float(vals[0]) != -999.0 else None
    r_mag = float(vals[1]) if vals[1] and float(vals[1]) != -999.0 else None
    gr = round(g - r_mag, 2) if g and r_mag else None
    return g, r_mag, gr

def skymapper_colors(ra, dec, radius_arcsec=2.0):
    """Query SkyMapper DR4 for nearest source g, r, v PSF mags via cone search."""
    radius_deg = radius_arcsec / 3600.0
    url = (f"https://skymapper.anu.edu.au/sm-cone/public/query"
           f"?RA={ra}&DEC={dec}&SR={radius_deg}&RESPONSEFORMAT=CSV&VERB=3")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    lines = r.text.strip().split('\n')
    if len(lines) < 2:
        return None, None, None, None, None
    header = lines[0].split(',')
    vals = lines[1].split(',')
    row = dict(zip(header, vals))
    def get_mag(key):
        v = row.get(key, '')
        return float(v) if v and v != 'NaN' else None
    g = get_mag('g_psf')
    r_mag = get_mag('r_psf')
    v = get_mag('v_psf')
    gr = round(g - r_mag, 2) if g and r_mag else None
    vg = round(v - g, 2) if v and g else None
    return g, r_mag, v, gr, vg

def hst_jwst_coverage(ra, dec, radius_arcsec=6.0):
    """Check for HST/JWST imaging at position. Returns dict of collections."""
    obs = Observations.query_criteria(
        coordinates=f"{ra} {dec}", radius=radius_arcsec / 3600.0,
        obs_collection=["HST", "JWST"], dataproduct_type="image")
    if len(obs) == 0:
        return {}
    result = {}
    for coll in set(obs["obs_collection"]):
        mask = obs["obs_collection"] == coll
        filters = sorted(set(obs["filters"][mask]))
        result[coll] = {"n_obs": int(mask.sum()), "filters": filters}
    return result

def nJy_to_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude."""
    if flux_nJy is None or flux_nJy <= 0:
        return None
    return -2.5 * np.log10(flux_nJy) + 31.4

def fmt_sigma(val, err):
    """Format value ± error with sigma = |val/err|."""
    if val is None or err is None or err == 0:
        return '—'
    sigma = abs(val / err)
    return f'{val:.2f} ± {err:.2f} ({sigma:.1f}σ)'

def add_scale_bar(img_bytes, pixscale_arcsec, bar_arcsec=1.0, fmt='jpeg'):
    """Draw a horizontal 1 arcsec scale bar on the lower-right of a cutout image."""
    img = Image.open(BytesIO(img_bytes)).convert('RGB')
    draw = ImageDraw.Draw(img)
    bar_px = int(round(bar_arcsec / pixscale_arcsec))
    margin = 5
    y = img.height - margin
    x_right = img.width - margin
    x_left = x_right - bar_px
    draw.line([(x_left, y), (x_right, y)], fill='red', width=1)
    buf = BytesIO()
    img.save(buf, format=fmt.upper())
    return buf.getvalue()

# --- UI ---
id_input = widgets.Textarea(
    value='ANT20267e97aeie6zfj',
    placeholder='Enter locus ID(s), one per line',
    description='Locus IDs:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px', height='60px')
)
counter_label = widgets.Label(value='')
prev_btn = widgets.Button(description='Prev', layout=widgets.Layout(width='60px'))
next_btn = widgets.Button(description='Next', layout=widgets.Layout(width='60px'))
fetch_btn = widgets.Button(description='Fetch')
output = widgets.Output()

# State for list navigation
_state = {'index': 0}

def get_id_list():
    """Parse textarea into list of non-empty IDs."""
    return [x.strip() for x in id_input.value.strip().split('\n') if x.strip()]

def show_locus(locus_id):
    """Display all vetting info for one locus."""
    try:
        locus = get_by_id(locus_id)
    except Exception as e:
        print(f'Error fetching locus: {e}')
        return

    # Basic info
    props = locus.properties or {}
    print(f'Locus ID: {locus.locus_id}')
    print(f'RA:       {locus.ra:.4f}')
    print(f'DEC:      {locus.dec:.4f}')
    # Lantern score
    max_score = props.get("lantern_xgboost_t2.0.7_c0.95_max_score")
    n_tagged = props.get("lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts")
    print(f'Lantern max_score:  {max_score}')
    print(f'Lantern num_tagged: {n_tagged}')


    # Clickable links
    antares_url = f'https://antares.noirlab.edu/loci/{locus.locus_id}'
    legacy_url = f'https://www.legacysurvey.org/viewer?ra={locus.ra:.6f}&dec={locus.dec:.6f}&layer=ls-dr10&zoom=16'
    display(HTML(
        f'<a href="{antares_url}" target="_blank">ANTARES page</a> | '
        f'<a href="{legacy_url}" target="_blank">Legacy Survey viewer</a>'
    ))

    # Latest alert info
    alerts = locus.alerts or []
    # Filter to LSST alerts only (skip ZTF)
    lsst_alerts = [a for a in alerts if any(k.startswith('lsst_') for k in (a.properties or {}))]
    if lsst_alerts:
        alerts = lsst_alerts
    if alerts:
        aprops = alerts[-1].properties or {}
        band = aprops.get("ant_passband") or aprops.get("passband") or "?"

        # Brightest/newest from locus properties (diaSource PSF flux = diff image)
        brightest = props.get("brightest_alert_magnitude")
        newest = props.get("newest_alert_magnitude")
        print(f'\nDiff mag (brightest): {brightest:.2f} ({band})' if brightest else '\nDiff mag (brightest): —')
        print(f'Diff mag (newest):   {newest:.2f} ({band})' if newest else 'Diff mag (newest):   —')

        # Science & template magnitudes from latest alert
        sci_flux = aprops.get("lsst_diaSource_scienceFlux")
        tmpl_flux = aprops.get("lsst_diaSource_templateFlux")
        diff_flux = aprops.get("lsst_diaSource_psfFlux")
        sci_mag = nJy_to_AB(sci_flux)
        tmpl_mag = nJy_to_AB(tmpl_flux)
        diff_mag = nJy_to_AB(diff_flux)
        print(f'Science mag (latest):  {sci_mag:.2f} ({band})  ({sci_flux:.2e} nJy)' if sci_mag else f'Science mag (latest):  —')
        print(f'Template mag (latest): {tmpl_mag:.2f} ({band})  ({tmpl_flux:.2e} nJy)' if tmpl_mag else f'Template mag (latest): —')
        print(f'Diff mag (latest):     {diff_mag:.2f} ({band})  ({diff_flux:.2e} nJy)' if diff_mag else f'Diff mag (latest):     —')

    # Catalog checks
    cats = list(locus.catalogs or [])
    print(f'\nCatalogs ({len(cats)}): {cats}')
    for c in ["milliquas", "gaia_dr3_variability", "gaia_dr3_gaia_source", "bright_guide_star_cat"]:
        print(f'  {c}: {"YES" if c in cats else "no"}')

    # Gaia variability
    if "gaia_dr3_variability" in cats:
        var_rows = locus.catalog_objects.get("gaia_dr3_variability", [])
        for i, row in enumerate(var_rows):
            vclass = row.get("class", "?")
            classifier = row.get("classifier", "?")
            print(f'  Variability {i}: class={vclass}, classifier={classifier}')

    # Milliquas
    if "milliquas" in cats:
        mq_rows = locus.catalog_objects.get("milliquas", [])
        for i, row in enumerate(mq_rows):
            name = row.get("name", "?")
            mtype = row.get("type", "?")
            z = row.get("z")
            rmag = row.get("rmag")
            z_str = f'{z:.3f}' if z else '\u2014'
            r_str = f'{rmag:.2f}' if rmag else '\u2014'
            print(f'  Milliquas {i}: {name}, type={mtype}, z={z_str}, r={r_str}')

    # Gaia proper motion, parallax, G mag, and BP-RP color
    if "gaia_dr3_gaia_source" in cats:
        gaia_rows = locus.catalog_objects.get("gaia_dr3_gaia_source", [])
        for i, row in enumerate(gaia_rows):
            g_mag = row.get("phot_g_mean_mag")
            bp_rp = row.get("bp_rp")
            print(f'  Gaia source {i}:')
            print(f'    G mag    = {g_mag:.2f}' if g_mag else '    G mag    = —')
            print(f'    BP-RP    = {bp_rp:.2f}' if bp_rp else '    BP-RP    = —')
            print(f'    pmra     = {fmt_sigma(row.get("pmra"), row.get("pmra_error"))} mas/yr')
            print(f'    pmdec    = {fmt_sigma(row.get("pmdec"), row.get("pmdec_error"))} mas/yr')
            print(f'    parallax = {fmt_sigma(row.get("parallax"), row.get("parallax_error"))} mas')

    # PS1 g-r color
    print('\nQuerying PS1 DR2...')
    try:
        ps1_g, ps1_r, ps1_gr = ps1_gr_color(locus.ra, locus.dec)
        if ps1_gr is not None:
            print(f'  PS1 g={ps1_g:.2f}, r={ps1_r:.2f}, g-r={ps1_gr:.2f}')
        else:
            print('  No PS1 match (or missing g/r).')
    except Exception as e:
        print(f'  PS1 query error: {e}')

    # SkyMapper g-r and v-g color
    print('Querying SkyMapper DR4...')
    try:
        sm_g, sm_r, sm_v, sm_gr, sm_vg = skymapper_colors(locus.ra, locus.dec)
        parts = []
        if sm_g: parts.append(f'g={sm_g:.2f}')
        if sm_r: parts.append(f'r={sm_r:.2f}')
        if sm_v: parts.append(f'v={sm_v:.2f}')
        if sm_gr is not None: parts.append(f'g-r={sm_gr:.2f}')
        if sm_vg is not None: parts.append(f'v-g={sm_vg:.2f}')
        if parts:
            print(f'  SkyMapper {", ".join(parts)}')
        else:
            print('  No SkyMapper match.')
    except Exception as e:
        print(f'  SkyMapper error: {e}')

    # HST/JWST coverage
    print('Checking HST/JWST coverage...')
    coverage = {}
    try:
        coverage = hst_jwst_coverage(locus.ra, locus.dec)
        if coverage:
            for coll, info in coverage.items():
                print(f'  {coll}: {info["n_obs"]} obs, filters: {", ".join(info["filters"])}')
        else:
            print('  No HST/JWST imaging.')
    except Exception as e:
        print(f'  MAST error: {e}')

    # Images side by side
    img_widgets = []

    # LSST difference thumbnail from the latest alert
    if alerts:
        alert_id = alerts[-1].alert_id
        print(f'\nFetching thumbnails for alert: {alert_id}')
        thumbs = get_thumbnails(alert_id) or {}
        for ttype, t in thumbs.items():
            if "diff" in str(ttype).lower():
                lsst_blob = add_scale_bar(t["blob"], 0.2, fmt='png')
                img_widgets.append(widgets.VBox([
                    widgets.Label('LSST Diff (6")'),
                    widgets.Image(value=lsst_blob,
                                  format='png', width=200, height=200)
                ]))
                break
        else:
            print(f'  No difference thumbnail. Types: {list(thumbs.keys())}')
    else:
        print('\nNo alerts on this locus.')

    # DECaLS cutout
    print('Fetching DECaLS cutout...')
    try:
        decals_blob = get_decals_jpg(locus.ra, locus.dec)
        decals_blob = add_scale_bar(decals_blob, 0.262)
        img_widgets.append(widgets.VBox([
            widgets.Label('DECaLS DR10 grz (26")'),
            widgets.Image(value=decals_blob, format='jpeg', width=200, height=200)
        ]))
    except Exception as e:
        print(f'  DECaLS error: {e}')

    # PS1 cutout
    print('Fetching PS1 cutout...')
    try:
        ps1_blob, ps1_label = get_ps1_color_cutout(locus.ra, locus.dec)
        if ps1_blob:
            ps1_blob = add_scale_bar(ps1_blob, 120 * 0.25 / 256)
            img_widgets.append(widgets.VBox([
                widgets.Label(f'PS1 ({ps1_label}, 30")'),
                widgets.Image(value=ps1_blob, format='jpeg', width=200, height=200)
            ]))
        else:
            print('  No PS1 coverage.')
    except Exception as e:
        print(f'  PS1 cutout error: {e}')

    # HST cutout (if coverage exists)
    if coverage and 'HST' in coverage:
        print('Fetching HST cutout...')
        try:
            hst_blob = get_hst_cutout(locus.ra, locus.dec)
            if hst_blob:
                hst_blob = add_scale_bar(hst_blob, 0.003 * 3600 / 256)
                img_widgets.append(widgets.VBox([
                    widgets.Label('HST (11")'),
                    widgets.Image(value=hst_blob, format='jpeg', width=200, height=200)
                ]))
            else:
                print('  HST cutout not available via HiPS.')
        except Exception as e:
            print(f'  HST cutout error: {e}')


    # JWST cutout (if coverage exists)
    if coverage and 'JWST' in coverage:
        print('Fetching JWST cutout...')
        try:
            jwst_blob = get_jwst_cutout(locus.ra, locus.dec)
            if jwst_blob:
                jwst_blob = add_scale_bar(jwst_blob, 0.003 * 3600 / 256)
                img_widgets.append(widgets.VBox([
                    widgets.Label('JWST (11")'),
                    widgets.Image(value=jwst_blob, format='jpeg', width=200, height=200)
                ]))
            else:
                print('  JWST cutout not available via HiPS.')
        except Exception as e:
            print(f'  JWST cutout error: {e}')

    # Euclid cutout
    print('Fetching Euclid cutout...')
    try:
        euclid_blob, euclid_label = get_euclid_cutout(locus.ra, locus.dec)
        if euclid_blob:
            _eimg = Image.open(BytesIO(euclid_blob))
            euclid_pxsc = 2 * 0.1 * 60 / _eimg.width
            euclid_blob = add_scale_bar(euclid_blob, euclid_pxsc, fmt='png')
            img_widgets.append(widgets.VBox([
                widgets.Label(f'Euclid ({euclid_label}, 12")'),
                widgets.Image(value=euclid_blob, format='png', width=200, height=200)
            ]))
        else:
            print('  No Euclid coverage.')
    except Exception as e:
        print(f'  Euclid error: {e}')

    if img_widgets:
        display(widgets.HBox(img_widgets))

def fetch_current(index=None):
    """Fetch and display the locus at current index."""
    ids = get_id_list()
    if not ids:
        with output:
            output.clear_output()
            print('Please enter at least one locus ID.')
        return
    if index is not None:
        _state['index'] = index
    _state['index'] = max(0, min(_state['index'], len(ids) - 1))
    counter_label.value = f'{_state["index"] + 1} / {len(ids)}'
    output.clear_output()
    with output:
        show_locus(ids[_state['index']])

def on_fetch(btn):
    _state['index'] = 0
    fetch_current()

def on_prev(btn):
    _state['index'] -= 1
    fetch_current()

def on_next(btn):
    _state['index'] += 1
    fetch_current()

fetch_btn.on_click(on_fetch)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
display(id_input,
        widgets.HBox([fetch_btn, prev_btn, next_btn, counter_label]),
        output)